# Out-of-sample validation 02

Freeze the Phase 1 sizing estimate using information available through November 4, 2024, then predict TAN's November 6 market-adjusted move:

\[
\widehat{r}^{MA}_{TAN,Nov\ 6} = \Delta E \times (1 - p_{Nov\ 4}).
\]

`ΔE` is the pre-specified event-window high-signal estimate from Phase 1. No November 5 or November 6 outcomes enter the prediction.

In [ ]:
from pathlib import Path

import pandas as pd

PHASE1_RESULTS_PATH = Path("data/01_estimate_delta_final_comparison.csv")
DAILY_DATA_PATH = Path("data/daily_merged_tan_spy_polymarket_2024.csv")
POLYMARKET_PATH = Path("data/polymarket_trump_2024_yes_1min.csv")
PREDICTION_PATH = Path("data/outofsamplevalidation02_nov6_prediction.csv")
CUTOFF_DATE = pd.Timestamp("2024-11-04")
ENTRY_TIMESTAMP = pd.Timestamp("2024-11-04T20:00:00Z")
TARGET_DATE = pd.Timestamp("2024-11-06")

In [ ]:
phase1_results = pd.read_csv(PHASE1_RESULTS_PATH)
daily = pd.read_csv(DAILY_DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
polymarket = (
    pd.read_csv(POLYMARKET_PATH, parse_dates=["timestamp_utc"])
    .set_index("timestamp_utc")
    .sort_index()
)

# Freeze inputs at the November 4 U.S. equity close (20:00 UTC).
phase1_daily = daily.loc[:CUTOFF_DATE].copy()
assert phase1_daily.index.max() == CUTOFF_DATE
assert TARGET_DATE not in phase1_daily.index

sizing_row = phase1_results.loc[
    phase1_results["role"].eq("Headline / sizing")
]
if len(sizing_row) != 1:
    raise RuntimeError(f"Expected one Phase 1 sizing estimate, found {len(sizing_row)}.")

delta_E = float(sizing_row.iloc[0]["delta_E"])
delta_E_se = float(sizing_row.iloc[0]["standard_error"])
p_entry_midpoint = float(polymarket["price"].asof(ENTRY_TIMESTAMP))

assert 0 < p_entry_midpoint < 1

{
    "phase1_cutoff": CUTOFF_DATE.date().isoformat(),
    "entry_timestamp_utc": ENTRY_TIMESTAMP.isoformat(),
    "delta_E_frozen": delta_E,
    "delta_E_se": delta_E_se,
    "p_entry_midpoint": p_entry_midpoint,
}

In [ ]:
remaining_probability_move = 1.0 - p_entry_midpoint
predicted_market_adjusted_move = delta_E * remaining_probability_move
predicted_se_from_delta_E = abs(remaining_probability_move) * delta_E_se

prediction = pd.DataFrame(
    {
        "phase1_cutoff": [CUTOFF_DATE.date().isoformat()],
        "entry_timestamp_utc": [ENTRY_TIMESTAMP.isoformat()],
        "target_date": [TARGET_DATE.date().isoformat()],
        "delta_E_frozen": [delta_E],
        "delta_E_se": [delta_E_se],
        "p_entry_midpoint": [p_entry_midpoint],
        "remaining_probability_move": [remaining_probability_move],
        "predicted_TAN_market_adjusted_move": [predicted_market_adjusted_move],
        "predicted_se_from_delta_E_only": [predicted_se_from_delta_E],
    }
)
prediction.to_csv(PREDICTION_PATH, index=False)

print(f"Frozen ΔE: {delta_E:.6f}")
print(f"Entry midpoint at 20:00 UTC: {p_entry_midpoint:.6f}")
print(f"1 − entry midpoint: {remaining_probability_move:.6f}")
print(
    "Predicted Nov 6 TAN market-adjusted move: "
    f"{predicted_market_adjusted_move:.6f} "
    f"({predicted_market_adjusted_move:.2%})"
)
print(f"Saved prediction to {PREDICTION_PATH.resolve()}")

prediction

## Frozen prediction

- Phase 1 cutoff: **November 4, 2024**
- Frozen event-window ΔE: **−0.234202**
- Entry timestamp: **November 4, 2024, 20:00 UTC** (U.S. equity close)
- Entry midpoint probability: **0.5785**
- Remaining probability move to resolution: **0.4215**
- **Predicted November 6 TAN market-adjusted move: −0.098716 (−9.87%)**

- SE propagated from ΔE only: **0.047510**

The propagated SE treats the 20:00 UTC midpoint as fixed and therefore does not include prediction-market measurement uncertainty.

## Realized out-of-sample comparison

Realized moves use adjusted-close returns and subtract SPY's corresponding return from TAN's return. Both horizons are anchored at the **November 4 close**, matching the frozen information set used by the prediction:

- One-day election-reaction endpoint: November 4 close → November 6 close
- Five-trading-day lag endpoint: November 4 close → November 12 close

This captures all repricing after the forecast was frozen, including November 5. For each horizon:

\[
\text{estimation error} = \frac{\text{realized} - \text{predicted}}{\text{predicted}}.
\]

Because the prediction is negative, a positive error means the realized decline was larger in magnitude than predicted.

In [ ]:
ETF_DATA_PATH = Path("data/tan_spy_daily_2016_2024.csv")
VALIDATION_RESULTS_PATH = Path("data/outofsamplevalidation02_realized_comparison.csv")
LAG_END_DATE = pd.Timestamp("2024-11-12")

etf_prices = pd.read_csv(ETF_DATA_PATH, parse_dates=["Date"])
adjusted_close = (
    etf_prices.pivot(index="Date", columns="Ticker", values="Adj Close")
    .sort_index()
)
etf_returns = adjusted_close.pct_change(fill_method=None)

# Both realized horizons begin at the Nov 4 close, matching the forecast origin.
entry_prices = adjusted_close.loc[CUTOFF_DATE, ["TAN", "SPY"]]
nov6_returns = (
    adjusted_close.loc[TARGET_DATE, ["TAN", "SPY"]] / entry_prices - 1
)
cumulative_returns = (
    adjusted_close.loc[LAG_END_DATE, ["TAN", "SPY"]] / entry_prices - 1
)

nov6_tan_return = nov6_returns["TAN"]
nov6_spy_return = nov6_returns["SPY"]
realized_nov6 = nov6_tan_return - nov6_spy_return
realized_nov6_12 = cumulative_returns["TAN"] - cumulative_returns["SPY"]

assert adjusted_close.loc[CUTOFF_DATE:LAG_END_DATE, ["TAN", "SPY"]].notna().all().all()
assert CUTOFF_DATE < TARGET_DATE < LAG_END_DATE

realized_nov6, realized_nov6_12

In [ ]:
validation = pd.DataFrame(
    [
        {
            "horizon": "Nov 4 close → Nov 6 close",
            "predicted_market_adjusted_move": predicted_market_adjusted_move,
            "realized_TAN_return": nov6_tan_return,
            "realized_SPY_return": nov6_spy_return,
            "realized_market_adjusted_move": realized_nov6,
        },
        {
            "horizon": "Nov 4 close → Nov 12 close",
            "predicted_market_adjusted_move": predicted_market_adjusted_move,
            "realized_TAN_return": cumulative_returns["TAN"],
            "realized_SPY_return": cumulative_returns["SPY"],
            "realized_market_adjusted_move": realized_nov6_12,
        },
    ]
)
validation["estimation_error"] = (
    validation["realized_market_adjusted_move"]
    - validation["predicted_market_adjusted_move"]
) / validation["predicted_market_adjusted_move"]

validation.to_csv(VALIDATION_RESULTS_PATH, index=False)

for row in validation.itertuples(index=False):
    print(
        f"{row.horizon}: predicted={row.predicted_market_adjusted_move:.2%}, "
        f"realized={row.realized_market_adjusted_move:.2%}, "
        f"estimation error={row.estimation_error:.2%}"
    )
print(f"Saved comparison to {VALIDATION_RESULTS_PATH.resolve()}")

validation

## Validation results

All realized returns begin at the November 4 close:

- Predicted market-adjusted move: **−9.87%**
- Realized through November 6: **−13.05%**
  - ΔE estimation error: **32.19%**
- Realized through November 12: **−20.24%**
  - Total realized deviation: **104.99%**

The sign was predicted correctly. The one-day election-reaction decline was 32.19% larger in magnitude than predicted, while the five-day decline was more than twice the predicted magnitude, consistent with substantial lagged equity adjustment.

## Final deliverable

Use the November 4 close → November 6 close comparison to evaluate the frozen **ΔE estimate**. The November 4 close → November 12 close result is the **total realized deviation** of the hedge forecast, which adds the later equity convergence/lag response. These must not be conflated.

In [ ]:
FINAL_DELIVERABLE_PATH = Path("data/outofsamplevalidation02_final_deliverable.csv")

one_day_row = validation.loc[
    validation["horizon"].eq("Nov 4 close → Nov 6 close")
].iloc[0]
five_day_row = validation.loc[
    validation["horizon"].eq("Nov 4 close → Nov 12 close")
].iloc[0]

delta_E_estimation_error_1day = one_day_row["estimation_error"]
total_realized_deviation_5day = five_day_row["estimation_error"]
lag_increment = (
    five_day_row["realized_market_adjusted_move"]
    - one_day_row["realized_market_adjusted_move"]
)
five_to_one_day_ratio = abs(
    five_day_row["realized_market_adjusted_move"]
    / one_day_row["realized_market_adjusted_move"]
)

final_deliverable = pd.DataFrame(
    {
        "delta_E_estimation_error_1day": [delta_E_estimation_error_1day],
        "total_realized_deviation_5day": [total_realized_deviation_5day],
        "predicted_move": [predicted_market_adjusted_move],
        "realized_1day_move": [one_day_row["realized_market_adjusted_move"]],
        "realized_5day_move": [five_day_row["realized_market_adjusted_move"]],
        "p_s_not_equal_p_lag_component": [lag_increment],
        "five_to_one_day_magnitude_ratio": [five_to_one_day_ratio],
    }
)
final_deliverable.to_csv(FINAL_DELIVERABLE_PATH, index=False)

print(f"ΔE estimation error (1-day): {delta_E_estimation_error_1day:.2%}")
print(f"Total realized deviation (5-day): {total_realized_deviation_5day:.2%}")
print(
    f"1-day vs 5-day realized: "
    f"{one_day_row['realized_market_adjusted_move']:.2%} vs "
    f"{five_day_row['realized_market_adjusted_move']:.2%}"
)
print(f"p_s ≠ p lag component: {lag_increment:.2%}")
print(f"5-day / 1-day magnitude: {five_to_one_day_ratio:.2f}×")
print(f"Saved final deliverable to {FINAL_DELIVERABLE_PATH.resolve()}")

final_deliverable

# Final result

- **ΔE estimation error: 32.19%** on the November 4 close → November 6 close basis (**−3.18 pp of Q**).
- **Total realized deviation: 104.99%** on the November 4 close → November 12 close basis.
- **p_s ≠ p convergence / equity-lag component: −7.19 pp of Q**, the majority of the five-day deviation.

Harmonized returns:

- November 4 close → November 6 close: **−13.05%**
- November 4 close → November 12 close: **−20.24%**
- Five-day magnitude relative to the immediate endpoint: **1.55×**

The common November 4 anchor includes all post-forecast repricing, including November 5. The dominant hedge failure mode is the delayed equity adjustment, not the ΔE estimate itself; this convention matches the Phase 4 decomposition.